# Musanga NB08 v4_001 — Is Musanga detectable from space?

**Scope:** quick, reproducible **V3.1-only** detectability experiment using frozen 50 m UAV-derived Musanga cover, and previously downloaded Sentinel-2 + 64-channel annual satellite embeddings. **No Earth Engine, raster resampling, GPU, regional inference or new downloads.**

**Design:** four strict leave-one-forest-out (LOFO) tests on **identical held-out cells**. Models: (M0) training-forest mean, (M1) annual Sentinel-2 + 3-year quarterly composites, (M2) annual embeddings, (M3) S2 + embeddings. All use the exact same fixed balanced training draw; one HGB fit per fold/model. Targets are continuous cover fractions (0–1); diagnostic thresholds 5%, 20%, 40% are **not separately fitted**.

**Primary claim:** predictive association / out-of-forest transfer, **not** uniquely identifiable Musanga spectral signatures. The older V3.1 UAV teacher is itself estimated (model-derived); satellite correlation may partly reflect secondary forest, age, canopy structure or disturbance. This experiment cannot prove species-specific remote sensing.

**Outputs:** frozen input audit, matched LOFO predictions (reusable), forest-equal summary, high-cover discrimination scores, within-forest spectral contrasts, and 3 figures.

**Run order:** Sections 00–04. CPU only. Cache persists in `Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06/17_NB08_V3_1_detectability`. **V3.1 source and old experiment outputs are read-only.**

## 00 — Configuration and exact cached reference audit

In [ ]:
%pip -q install pyarrow scikit-learn scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
ROOTS=[
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
 Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
found=[p for p in ROOTS if (p/'frozen_50m_reference.parquet').is_file()]
assert len(found)==1,('Expected one existing frozen reference; no new downloads',found)
ROOT=found[0];OUT=ROOT/'17_NB08_V3_1_detectability'
OUT.mkdir(parents=True,exist_ok=True)
D=pd.read_parquet(ROOT/'frozen_50m_reference.parquet')
SITES=('Baego','MOS','Yangambi','Yoko')
KEYS=['site','mosaic','row','col']
assert len(D)==14454 and not D.duplicated(KEYS).any()
assert D.groupby('site').size().to_dict()=={'Baego':480,'MOS':11494,'Yangambi':1383,'Yoko':1097}
assert D.target_fraction.between(0,1).all()
assert {'x','y','target_fraction','site'}.issubset(D.columns)
ANNUAL=sorted(c for c in D if c.startswith('RAW_ANNUAL__'))
QUARTER=sorted(c for c in D if c.startswith('S2_QUARTERLY_3YR__'))
EMB=sorted(c for c in D if c.startswith('EMB_ANNUAL__'))
assert len(ANNUAL)>0 and len(QUARTER)==44 and len(EMB)==64,(len(ANNUAL),len(QUARTER),len(EMB))
# Quarterly clear-observation counts are not spectral reflectance.
Q_COUNTS=[f'S2_QUARTERLY_3YR__{b:02d}' for b in (11,22,33,44)]
assert set(Q_COUNTS).issubset(QUARTER)
S2=[*ANNUAL,*[c for c in QUARTER if c not in Q_COUNTS]]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':[ *S2,*EMB ]}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
assert np.isfinite(D[ALL_FEATURES+['target_fraction','x','y']].to_numpy('float32')).all()
assert (D[Q_COUNTS].to_numpy(float)>=1).all(), 'Frozen reference has quarterly no-observation cells; inspect source.'
SEED=20261120
CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
           min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
# Compute fingerprints once; do not overwrite unrelated historical outputs.
SOURCE_SHA=hashlib.sha256(pd.util.hash_pandas_object(
 D[KEYS+['x','y','target_fraction']+ALL_FEATURES],index=False).values.tobytes()).hexdigest()
CONFIG={'version':'NB08_V3_1_LOFO_detectability_v4_001','sites':SITES,
 'n':len(D),'features':{k:len(v) for k,v in FEATURE_SETS.items()},
 'seed':SEED,'cap':CAP_PER_TRAIN_FOREST,'model':MODEL,
 'thresholds':THRESHOLDS,'source_sha256':SOURCE_SHA}
SIGNATURE=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:14]
RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
config_path=RUN/'config.json'
if config_path.is_file():assert json.loads(config_path.read_text())==json.loads(json.dumps(CONFIG))
else:config_path.write_text(json.dumps(CONFIG,indent=2))
print('Source:',ROOT/'frozen_50m_reference.parquet')
print('Saved outputs:',RUN,'| signature:',SIGNATURE)
display(D.groupby('site').target_fraction.agg(n='size',mean='mean',std='std',maximum='max').round(3))
print('Predictors: S2 annual',len(ANNUAL),'+ quarterly reflectance',len(QUARTER)-len(Q_COUNTS),'; embeddings',len(EMB))
print('No imagery downloads or raster reads are required.')


## 01 — A strict, small, cached leave-one-forest-out experiment

Use the original V3.1 source's **balanced-by-forest and cover-stratum** training principle; training selections depend only on training-forest targets. Each held-out forest is untouched until scoring. Single-model fits are intended as a first detectability test rather than hyperparameter optimization.

In [ ]:
y=D.target_fraction.to_numpy(np.float32)
site=D.site.to_numpy()
def balanced_draw(pool,heldout):
 rng=np.random.default_rng(SEED+sum(map(ord,heldout)))
 chosen=[]
 for forest in SITES:
  ix=pool[site[pool]==forest]
  if not len(ix):continue
  cap=min(len(ix),CAP_PER_TRAIN_FOREST)
  if len(ix)<=cap:
   chosen.extend(ix.tolist());continue
  strata=np.digitize(y[ix],[.10,.20,.40])
  bins={j:ix[strata==j] for j in range(4)}
  pick=[]
  # Keep up to a quarter of training sample above 40% where available.
  take=min(len(bins[3]),int(np.ceil(.25*cap)))
  if take:
   ids=rng.choice(bins[3],take,replace=False)
   pick.extend(ids.tolist());bins[3]=np.setdiff1d(bins[3],ids)
  while len(pick)<cap:
   active=[k for k in bins if len(bins[k])]
   assert active
   for k in active:
    n=min(len(bins[k]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
    if n>0:
     ids=rng.choice(bins[k],n,replace=False)
     pick.extend(ids.tolist());bins[k]=np.setdiff1d(bins[k],ids)
    if len(pick)>=cap:break
  chosen.extend(pick)
 return np.asarray(sorted(chosen),dtype=int)
predictions=[]
audit=[]
for held in SITES:
 test=np.flatnonzero(site==held)
 train_pool=np.flatnonzero(site!=held)
 train=balanced_draw(train_pool,held)
 assert not np.intersect1d(train,test).size
 audit.append(dict(heldout=held,train_cells=len(train),test_cells=len(test),
                   high_train=int((y[train]>=.40).sum())))
 # The constant null is learned from the **same** training draw.
 null_value=float(np.mean(y[train]))
 frame=D.iloc[test][KEYS+['x','y','target_fraction']].copy().reset_index(drop=True)
 frame['heldout']=held
 frame['M0_mean']=null_value
 for model_name,cols in FEATURE_SETS.items():
  cache=RUN/f'{held}_{model_name}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test_index'],test)
    assert z['source_sha256'].item()==SOURCE_SHA
    pred=z['pred'].astype(np.float32)
   print('CACHE',held,model_name,flush=True)
  else:
   t=time.monotonic()
   model=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   model.fit(D.iloc[train][cols].to_numpy('float32'),y[train])
   pred=np.clip(model.predict(D.iloc[test][cols].to_numpy('float32')),0,1).astype(np.float32)
   tmp=cache.with_suffix('.partial')
   with open(tmp,'wb') as f:
    np.savez_compressed(f,test_index=test,pred=pred,source_sha256=SOURCE_SHA)
   os.replace(tmp,cache)
   print('FIT',held,model_name,'seconds',round(time.monotonic()-t,1),flush=True)
  frame[model_name]=pred
 predictions.append(frame)
P=pd.concat(predictions,ignore_index=True)
assert len(P)==len(D) and not P.duplicated(KEYS).any()
assert P[['M0_mean',*FEATURE_SETS]].notna().all().all()
P.to_parquet(RUN/'01_paired_LOFO_predictions.parquet',index=False)
pd.DataFrame(audit).to_csv(RUN/'01_training_fold_sizes.csv',index=False)
display(pd.DataFrame(audit))
print('All held-out predictions cached on the same 14,454 frozen reference cells.')


## 02 — Forest-equal continuous-cover prediction and abundance detection

Use per-forest equal-weight metrics, never global pixel weighting that allows MOS to dominate. Continuous regression: R², MAE, RMSE, bias and Spearman. For thresholds 5%, 20%, 40%, compute ROC-AUC and average precision from **continuous prediction scores**, plus prevalence and a training-derived constant-score baseline. Undefined AUC for missing positive or negative classes is reported as NaN (not zero).

In [ ]:
MODELS=['M0_mean',*FEATURE_SETS]
def metrics(g,col):
 obs=g.target_fraction.to_numpy(float)
 pred=g[col].to_numpy(float)
 err=pred-obs
 var=float(np.mean((obs-np.mean(obs))**2))
 rho=spearmanr(obs,pred).statistic if np.std(pred)>1e-12 and np.std(obs)>1e-12 else np.nan
 return dict(n=len(g),R2=float(1-np.mean(err**2)/var) if var>0 else np.nan,
             MAE_pp=float(100*np.mean(np.abs(err))),
             RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
             bias_pp=float(100*np.mean(err)),
             spearman_rho=float(rho) if np.isfinite(rho) else np.nan)
PER=pd.DataFrame([dict(site=forest,model=model,**metrics(g,model))
 for forest,g in P.groupby('site') for model in MODELS])
PER.to_csv(RUN/'02_continuous_by_forest.csv',index=False)
SUMMARY=[]
for model in MODELS:
 sub=PER[PER.model.eq(model)]
 # Global equal-forest R² with pooled equal-forest target variance.
 means=P.groupby('site').target_fraction.mean()
 v=np.mean([np.mean((g.target_fraction-means[s])**2)+(means[s]-means.mean())**2
      for s,g in P.groupby('site')])
 mse=np.mean([np.mean((g[model]-g.target_fraction)**2) for s,g in P.groupby('site')])
 SUMMARY.append(dict(model=model,n_cells=len(P),n_forests=len(SITES),
  equal_forest_R2=float(1-mse/v) if v>0 else np.nan,
  equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
  equal_forest_RMSE_pp=float(np.sqrt(np.mean(sub.RMSE_pp**2))),
  equal_forest_bias_pp=float(sub.bias_pp.mean()),
  mean_forest_rho=float(sub.spearman_rho.mean())))
SUMMARY=pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN/'02_equal_forest_summary.csv',index=False)
DISC=[]
for (forest,model),g in (( (f,m),q) for f,q in P.groupby('site') for m in MODELS):
 obs=g.target_fraction.to_numpy(float)
 pred=g[model].to_numpy(float)
 for t in THRESHOLDS:
  true=(obs>=t).astype(int)
  npos=int(true.sum());nneg=len(true)-npos
  auc=roc_auc_score(true,pred) if npos and nneg and np.unique(pred).size>1 else np.nan
  ap=average_precision_score(true,pred) if npos and nneg else np.nan
  DISC.append(dict(site=forest,model=model,threshold=t,
       n=len(g),positive_n=npos,negative_n=nneg,
       prevalence=npos/len(g),ROC_AUC=auc,average_precision=ap))
DISC=pd.DataFrame(DISC)
DISC.to_csv(RUN/'02_abundance_detection_by_forest.csv',index=False)
DISPLAY_DISC=(DISC.groupby(['model','threshold']).agg(
 n_forests_evaluable=('ROC_AUC','count'),
 mean_AUC=('ROC_AUC','mean'),
 mean_AP=('average_precision','mean'),
 mean_prevalence=('prevalence','mean')).reset_index())
DISPLAY_DISC.to_csv(RUN/'02_equal_forest_detection_summary.csv',index=False)
print('Equal-forest continuous-cover scores:')
display(SUMMARY.round(3))
print('Abundance discrimination (undefined sites omitted, counts explicit):')
display(DISPLAY_DISC.round(3))
print('Important: high-cover detection is undefined for forests without high-cover examples.')


## 03 — Visual evidence: spectral contrasts, held-out predictions and detectability

**Figure A:** within-forest low vs high cover spectral contrast on the frozen 50 m cells; avoids visually conflating broad differences among forests. Spectral columns are raw annual source bands (band numbers until independently verified); do not claim wavelengths without an audited metadata lookup.

**Figure B:** held-out continuous fractional cover by forest, preserving the four-fold transfer test.

**Figure C:** abundance discrimination curves at 5, 20 and 40% (mean of evaluable forests, number of evaluable forests displayed).

In [ ]:
import matplotlib
from matplotlib.lines import Line2D
# A: Within-forest annual S2 contrast. Use percentiles for robustness.
low=.05;high=.40
rows=[]
for forest,g in D.groupby('site'):
 lo=g[g.target_fraction<=low]
 hi=g[g.target_fraction>=high]
 if len(lo)<10 or len(hi)<10:
  print('Spectral contrast not estimable',forest,'low',len(lo),'high',len(hi));continue
 for i,c in enumerate(ANNUAL,1):
  lv=float(lo[c].median());hv=float(hi[c].median())
  rows.append(dict(site=forest,band=c,band_order=i,n_low=len(lo),
                   n_high=len(hi),median_low=lv,median_high=hv,
                   difference=hv-lv))
SPECTRA=pd.DataFrame(rows)
SPECTRA.to_csv(RUN/'03_within_forest_annual_S2_spectra.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=SPECTRA[SPECTRA.site.eq(forest)]
 if len(g):
  ax.plot(g.band_order,g.median_low,'o-',label='≤5% UAV Musanga')
  ax.plot(g.band_order,g.median_high,'o-',label='≥40% UAV Musanga')
  ax.set_title(f'{forest} | low n={g.n_low.iloc[0]}, high n={g.n_high.iloc[0]}')
 else:
  ax.set_title(f'{forest} | too few high/low cells')
 ax.set(xlabel='Annual S2 source band number',ylabel='Median archived band value')
 ax.grid(alpha=.15)
axs.flat[0].legend(fontsize=8)
fig.suptitle('A · Within-forest spectral associations (not species-specific identification)')
fig.savefig(RUN/'03A_S2_within_forest_low_high_spectra.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)
# B: For each forest show S2 vs embeddings+S2; cap plots only, not metrics.
fig,axs=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=P[P.site.eq(forest)].copy()
 if len(g)>1300:g=g.sample(n=1300,random_state=SEED)
 ax.scatter(g.target_fraction*100,g.M1_S2*100,s=9,alpha=.3,label='S2')
 ax.scatter(g.target_fraction*100,g.M3_S2_EMB*100,s=9,alpha=.3,label='S2 + embeddings')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlabel='UAV Musanga cover (%)',
        ylabel='Held-out satellite prediction (%)',
        xlim=(-2,102),ylim=(-2,102))
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('B · Strict leave-one-forest-out prediction (matched reference cells)')
fig.savefig(RUN/'03B_heldout_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# C: Mean AUC by cover threshold and number of forests actually evaluable.
fig,ax=plt.subplots(figsize=(8,5))
for model in MODELS[1:]:
 q=DISPLAY_DISC[DISPLAY_DISC.model.eq(model)].sort_values('threshold')
 ax.plot(q.threshold*100,q.mean_AUC,marker='o',label=model)
 for v in q.itertuples():
  if np.isfinite(v.mean_AUC):
   ax.annotate(f'n={v.n_forests_evaluable}',(v.threshold*100,v.mean_AUC),
               xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
ax.axhline(.5,ls='--',color='gray',lw=1,label='Chance AUC')
ax.set(xlabel='UAV Musanga cover threshold (%)',ylabel='Forest-mean held-out ROC-AUC',
       ylim=(0,1.05),xticks=[5,20,40],
       title='C · Ability to distinguish Musanga-abundant patches')
ax.legend(fontsize=9);ax.grid(alpha=.15)
fig.savefig(RUN/'03C_abundance_discrimination.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Three figures saved:',RUN)


## 04 — Compact decision and caveats

Interpret performance for **all four forests**, not just the pooled R². Compare M1 vs M0 (satellite information); M2 vs M1 (embeddings versus raw S2); and M3 vs both. AUC > 0.5 on evaluable, held-out forests supports **association**, not Musanga-specific spectral identifiability. Inspect forest abundance prevalence: at >40% the Yangambi fold has no positives, so the ROC-AUC is not defined.

**Caveats:** frozen V3.1 labels derive from an earlier UAV model; pixel-neighbour dependence invalidates naive cell-wise confidence intervals. Sampling to balance training forests is fixed, not evaluated as an optimal algorithm. The threshold experiment uses continuous regression scores—not binary models—avoiding separate threshold-fitting. Do not interpret within-forest band profiles as independent species spectra; examine confounding by young forest and disturbance in a later experiment.

**Next iteration:** once NB07 V4 RGB+MS inference is complete, reaggregate its valid-pixel predictions to the exact 50 m satellite cells, keep a matched validity mask, and repeat the same fixed experiment. Do not mix V3.1 and V4 targets in a single score.

In [ ]:
manifest={'experiment':'NB08_v4_001_V3_1','input':str(ROOT/'frozen_50m_reference.parquet'),
 'source_sha256':SOURCE_SHA,'signature':SIGNATURE,
 'n_reference_cells':len(D),'n_sites':len(SITES),
 'feature_families':{k:len(v) for k,v in FEATURE_SETS.items()},
 'spatial_validation':'leave_one_forest_out',
 'reference':'V3_1 UAV teacher-derived fractional Musanga cover',
 'warning':'Predictive association is not species-unique identification',
 'outputs':str(RUN)}
(RUN/'04_run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('NB08 V3.1 detection experiment completed:',RUN)
print('Primary output: 02_equal_forest_summary.csv')
print('Detection output: 02_equal_forest_detection_summary.csv')
print('Paired predictions: 01_paired_LOFO_predictions.parquet')
